In [ ]:
import ollama
import os
import pandas as pd
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
DATA_PATH = "Legal"

loader = DirectoryLoader(
    DATA_PATH,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

print("Total pages/documents loaded:", len(documents))

# Check metadata
for doc in documents[:3]:
    print(doc.metadata)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_documents = text_splitter.split_documents(documents)

# Add a unique chunk ID
for i, doc in enumerate(split_documents):
    doc.metadata["chunk_id"] = i

print("Total chunks created:", len(split_documents))

print("\nSample chunk:")
print(split_documents[0].page_content[:500])
print("\nMetadata:", split_documents[0].metadata)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

In [ ]:
vector_store = FAISS.from_documents(
    documents=split_documents,
    embedding=embeddings
)

print("FAISS vector store created successfully!")
print("Total vectors:", vector_store.index.ntotal)

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

print("Retriever created successfully!")

In [ ]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
You are a legal document assistant.

Answer the question ONLY using the information provided in the context.

Rules:
- Do not use outside knowledge.
- Do not make assumptions.
- If the answer is not available in the context, say:
  "Not found in the provided legal documents."

Context:
{context}

Question:
{question}

Answer:
""")

print("LLM and prompt created successfully!")

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded successfully!")

In [ ]:
def run_langchain_rag(question, initial_k=5, final_k=3):

    # -----------------------------------
    # STEP 1: Retrieve initial documents
    # -----------------------------------
    
    initial_docs = retriever.invoke(question)

    # -----------------------------------
    # STEP 2: Rerank retrieved documents
    # -----------------------------------
    
    pairs = [
        (question, doc.page_content)
        for doc in initial_docs
    ]

    scores = reranker.predict(pairs)

    ranked_results = sorted(
        zip(scores, initial_docs),
        key=lambda x: x[0],
        reverse=True
    )

    # Select final top documents
    final_docs = [
        doc for score, doc in ranked_results[:final_k]
    ]

    final_scores = [
        float(score)
        for score, doc in ranked_results[:final_k]
    ]

    # -----------------------------------
    # STEP 3: Create context
    # -----------------------------------
    
    context = "\n\n".join(
        doc.page_content
        for doc in final_docs
    )

    # -----------------------------------
    # STEP 4: Create prompt
    # -----------------------------------
    
    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    # -----------------------------------
    # STEP 5: Generate answer
    # -----------------------------------
    
    response = llm.invoke(formatted_prompt)

    answer = response.content

    return {
        "answer": answer,
        "retrieved_documents": final_docs,
        "reranking_scores": final_scores,
        "context": context
    }

In [ ]:
test_questions = [

    {
        "question": "What evidence suggests that the disputed delivery challan in the Chowdary case may have been forged?",
        "expected_source": "01_Criminal_Case_File_Chowdary.pdf"
    },

    {
        "question": "What was the main legal issue regarding the accused's intention in the criminal case?",
        "expected_source": "01_Criminal_Case_File_Chowdary.pdf"
    },

    {
        "question": "Why did BlueWave Constructions terminate its contract with Orion Infra Solutions?",
        "expected_source": "02_Commercial_Dispute_Case_File_BlueWave.pdf"
    },

    {
        "question": "What were the arguments of the parties regarding force majeure in the BlueWave commercial dispute?",
        "expected_source": "02_Commercial_Dispute_Case_File_BlueWave.pdf"
    },

    {
        "question": "What did the Survey Commissioner's report conclude about the disputed 150 square yards in the Lakshmi Devi property case?",
        "expected_source": "03_Property_Dispute_Case_File_LakshmiDevi.pdf"
    },

    {
        "question": "Why does Lakshmi Devi challenge the validity of the defendant's 2021 sale deed?",
        "expected_source": "03_Property_Dispute_Case_File_LakshmiDevi.pdf"
    },

    {
        "question": "Why does Anil Kumar claim that his termination was unfair or targeted?",
        "expected_source": "04_Employment_Tribunal_Case_File_AnilKumar.pdf"
    },

    {
        "question": "What reasons did Vertex Software Solutions give for retaining two junior employees during the workforce reduction?",
        "expected_source": "04_Employment_Tribunal_Case_File_AnilKumar.pdf"
    },

    {
        "question": "What is the capital city of France?",
        "expected_source": None
    }
]

In [ ]:
def calculate_faithfulness(answer, context):

    faithfulness_prompt = f"""
Is the answer completely supported by the given context?

Context:
{context}

Answer:
{answer}

Return ONLY one number:
1 = The answer is supported by the context.
0 = The answer is not supported by the context.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": faithfulness_prompt
            }
        ],
        options={
            "temperature": 0
        }
    )

    result = response["message"]["content"].strip()

    if result.startswith("1"):
        return 1
    else:
        return 0

In [ ]:
def calculate_answer_relevance(question, answer):

    relevance_prompt = f"""
Does the answer directly answer the question?

Question:
{question}

Answer:
{answer}

Return ONLY one number:
1 = Yes, the answer directly answers the question.
0 = No, the answer does not directly answer the question.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "user",
                "content": relevance_prompt
            }
        ],
        options={
            "temperature": 0
        }
    )

    result = response["message"]["content"].strip()

    if result.startswith("1"):
        return 1
    else:
        return 0

In [ ]:
results = []

k = 3

for question_number, item in enumerate(test_questions, start=1):

    question = item["question"]
    expected_source = item["expected_source"]

    print("\n" + "=" * 100)
    print(f"QUESTION {question_number}")
    print("=" * 100)
    print(question)

    # ----------------------------------
    # RUN LANGCHAIN RAG
    # ----------------------------------

    rag_result = run_langchain_rag(
        question,
        initial_k=5,
        final_k=k
    )

    answer = rag_result["answer"]
    retrieved_documents = rag_result["retrieved_documents"]
    context = rag_result["context"]

    # ----------------------------------
    # GET RETRIEVED SOURCES
    # ----------------------------------

    retrieved_sources = []

    for doc in retrieved_documents:

        source_path = doc.metadata.get(
            "source",
            ""
        )

        source_name = os.path.basename(
            source_path
        )

        retrieved_sources.append(source_name)

    # Remove duplicates
    unique_sources = list(
        dict.fromkeys(retrieved_sources)
    )

    # ----------------------------------
    # HIT RATE
    # ----------------------------------

    if expected_source is not None:

        hit_rate = int(
            expected_source in retrieved_sources
        )

    else:
        hit_rate = None

    # ----------------------------------
    # PRECISION@K
    # ----------------------------------

    if expected_source is not None:

        relevant_chunks = sum(
            source == expected_source
            for source in retrieved_sources
        )

        precision_at_k = (
            relevant_chunks / len(retrieved_sources)
        )

    else:
        precision_at_k = None

    # ----------------------------------
    # RECALL@K
    # ----------------------------------

    if expected_source is not None:

        recall_at_k = int(
            expected_source in retrieved_sources
        )

    else:
        recall_at_k = None

    # ----------------------------------
    # FAITHFULNESS
    # ----------------------------------

    faithfulness = calculate_faithfulness(
        answer,
        context
    )

    # ----------------------------------
    # ANSWER RELEVANCE
    # ----------------------------------

    answer_relevance = calculate_answer_relevance(
        question,
        answer
    )

    # ----------------------------------
    # STORE RESULTS
    # ----------------------------------

    result = {

        "question_number": question_number,

        "question": question,

        "expected_source": expected_source,

        "retrieved_sources": unique_sources,

        "reranking_scores": rag_result["reranking_scores"],

        "answer": answer,

        "hit_rate": hit_rate,

        f"precision@{k}": precision_at_k,

        f"recall@{k}": recall_at_k,

        "faithfulness": faithfulness,

        "answer_relevance": answer_relevance
    }

    results.append(result)

    # ----------------------------------
    # PRINT RESULTS
    # ----------------------------------

    print("\nEXPECTED SOURCE:")
    print(expected_source)

    print("\nRETRIEVED SOURCES:")
    for source in unique_sources:
        print("-", source)

    print("\nANSWER:")
    print(answer)

    print("\nMETRICS:")
    print("Hit Rate         :", hit_rate)
    print(f"Precision@{k}      :", precision_at_k)
    print(f"Recall@{k}         :", recall_at_k)
    print("Faithfulness      :", faithfulness)
    print("Answer Relevance  :", answer_relevance)


print("\n" + "=" * 100)
print("ALL QUESTIONS COMPLETED")
print("Total results stored:", len(results))
print("=" * 100)

In [ ]:
results_df = pd.DataFrame(results)

results_df[
    [
        "question_number",
        "question",
        "expected_source",
        "retrieved_sources",
        "hit_rate",
        "precision@3",
        "recall@3",
        "faithfulness",
        "answer_relevance"
    ]
]